# 3D manual mock: assemble a triangle from 3 sticks

A top-down mock of the **action-grounded, state-aware** manual. Instead of generating pictures, the
manual operates on **3D objects with 6-DoF poses** and shows, at each step, the *action*: which stick
moves, where its end mates, and how it rotates into place.

- Objects are real 3D entities (you can **rotate / zoom** the scene with the mouse).
- Press **Play** to run the manual: stick A becomes the base, B mates at vertex V1, C closes the triangle.
- Each step states the **action and the connection it creates** (the assembly graph), not just a picture.

This is the ideal-representation idea on a toy: state = stick poses + which ends are joined;
step = a mating action between an end and a vertex. No GPU, no model - pure geometry.

In [ ]:
!pip install -q plotly numpy

: 

In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Geometry: equilateral triangle, side 2, half-stick length 1 ---
V0 = np.array([0.0, 0.0, 0.0])
V1 = np.array([2.0, 0.0, 0.0])
V2 = np.array([1.0, 1.7320508, 0.0])
H = 1.0  # half length

# Each stick = (center, angle_deg). Targets place them on the triangle edges.
TARGET = {
    "A": (np.array([1.0, 0.0, 0.0]),       0.0),   # base  V0-V1
    "B": (np.array([1.5, 0.8660254, 0.0]), 120.0), # right V1-V2
    "C": (np.array([0.5, 0.8660254, 0.0]), 240.0), # left  V2-V0
}
# Loose starting poses (scattered around)
START = {
    "A": (np.array([-3.0, -2.0, 0.0]),  10.0),
    "B": (np.array([ 4.5, -0.5, 0.0]),  80.0),
    "C": (np.array([-3.5,  3.5, 0.0]), 200.0),
}
ORDER = ["A", "B", "C"]

def endpoints(center, ang_deg):
    th = np.radians(ang_deg)
    d = np.array([np.cos(th), np.sin(th), 0.0])
    return center - H*d, center + H*d

def pose_at(name, phase, t):
    # phase = index of the stick currently being placed; t = progress 0..1
    i = ORDER.index(name)
    if i < phase:                      # already placed
        return TARGET[name], "placed"
    if i > phase:                      # still loose
        return START[name], "waiting"
    (c0, a0), (c1, a1) = START[name], TARGET[name]  # active: interpolate
    return (c0 + (c1 - c0)*t, a0 + (a1 - a0)*t), "active"

COLOR = {"placed": "#2ca02c", "active": "#ff7f0e", "waiting": "#c8c8c8"}

ACTIONS = [
    "Step 1 / 3   |   Place stick A as the base.   Connections: (none yet)",
    "Step 2 / 3   |   Take stick B. Mate its end to vertex V1 (free end of A), swing to 120 deg.   Connections: A-B @ V1",
    "Step 3 / 3   |   Take stick C. Mate one end to V2 (free end of B), the other to V0 (free end of A).   Connections: A-B, B-C, C-A  -> triangle closed",
]
HILITE = [[], [V1], [V2, V0]]  # vertices being mated in each step

def build_traces(phase, t):
    sticks = []
    ep_x, ep_y, ep_z = [], [], []
    for name in ORDER:
        (c, a), state = pose_at(name, phase, t)
        p0, p1 = endpoints(c, a)
        sticks.append(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode="lines+text", line=dict(width=14, color=COLOR[state]),
            text=["", name], textposition="top center", showlegend=False))
        for p in (p0, p1):
            ep_x.append(p[0]); ep_y.append(p[1]); ep_z.append(p[2])
    ends = go.Scatter3d(x=ep_x, y=ep_y, z=ep_z, mode="markers",
                        marker=dict(size=4, color="#333"), showlegend=False)
    hv = HILITE[phase]
    verts = go.Scatter3d(
        x=[V0[0], V1[0], V2[0]], y=[V0[1], V1[1], V2[1]], z=[V0[2], V1[2], V2[2]],
        mode="markers",
        marker=dict(size=[14 if any(np.allclose(V, h) for h in hv) else 6 for V in (V0, V1, V2)],
                    color=["#d62728" if any(np.allclose(V, h) for h in hv) else "#999" for V in (V0, V1, V2)]),
        showlegend=False)
    return sticks + [ends, verts]

# --- Build animation frames ---
PER = 16
frames, slider_steps = [], []
k = 0
for phase in range(3):
    for f in range(PER + 1):
        t = f/PER
        name = "f%d" % k
        frames.append(go.Frame(data=build_traces(phase, t), name=name,
                                layout=go.Layout(title=ACTIONS[phase])))
        slider_steps.append(dict(args=[[name], dict(frame=dict(duration=0, redraw=True), mode="immediate")],
                                 label="", method="animate"))
        k += 1

AX = dict(range=[-5, 6], showbackground=True, backgroundcolor="#f5f5f5")
fig = go.Figure(
    data=build_traces(0, 0.0),
    layout=go.Layout(
        title=ACTIONS[0],
        scene=dict(xaxis=dict(AX), yaxis=dict(range=[-3, 4]), zaxis=dict(range=[-1.5, 1.5]),
                   aspectmode="data"),
        width=900, height=650,
        updatemenus=[dict(type="buttons", showactive=False, y=1, x=0,
            buttons=[dict(label="Play", method="animate",
                          args=[None, dict(frame=dict(duration=70, redraw=True), fromcurrent=True)]),
                     dict(label="Pause", method="animate",
                          args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(active=0, steps=slider_steps, x=0.1, len=0.85, currentvalue=dict(visible=False))]),
    frames=frames)
fig.show()

: 

## How this maps to the real manual idea

- **Operate on 3D objects, not pixels.** Each stick is a 6-DoF rigid body; rotate/zoom to inspect from
  any angle. Rendering is a *consequence* of the geometry, so it never drifts or hallucinates.
- **State, not a slideshow.** The system knows each stick's pose and which ends are joined. That is the
  "sense of current structure" flipbook lacks (limitation 01).
- **Action-based steps.** Each step is an operator: *which* stick moves, *which* end mates to *which*
  vertex, and the rotation to get there - shown as a trajectory, plus the connection it creates
  (limitation 02). The red vertex marks the mating site for that step.
- **Verify, then advance.** In a real build you would confirm the stick reached its target pose before
  the next step. Here the manual just plays through; swap in pose estimation to gate it.

Scale the same skeleton up: replace 3 sticks with IKEA parts (CAD models), the hardcoded target poses
with the manual's per-step poses, and the scripted motion with a learned next-action predictor.